In [16]:
import numpy as np
from tqdm.auto import tqdm

from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import VectorSearch, Index

embed = Embedder()

Q1

In [17]:
v = embed.encode("How does approximate nearest neighbor search work?")
print(f"len(v) = {len(v)}")
print(f"v[0] = {v[0]:.4f}")

len(v) = 384
v[0] = -0.0206


In [18]:
v[0]

np.float64(-0.02058203437252893)

Q2

In [19]:
reader = GithubRepositoryDataReader(  # создаём ридер для чтения файлов из GitHub репозитория
    repo_owner="DataTalksClub",       # владелец репозитория (организация или юзер)
    repo_name="llm-zoomcamp",         # название репозитория
    commit_id="8c1834d",              # конкретный коммит — фиксируем версию, чтобы результаты были воспроизводимы
    allowed_extensions={"md"},        # читаем только .md файлы (Markdown документация)
    filename_filter=lambda path: "/lessons/" in path,  # фильтр: берём только файлы из папки /lessons/
)
files = reader.read()  # скачиваем/читаем файлы из репо по заданным параметрам

documents = [file.parse() for file in files]  # парсим каждый файл в словарь {'filename', 'content'}
documents[0]  # смотрим на первый документ — проверка что всё прочиталось корректно

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [20]:
len(documents)

72

In [21]:
target_name = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next(d for d in documents if d["filename"] == target_name)

In [22]:
[d for d in documents if d["filename"] == target_name]

[{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done

In [23]:
next(d for d in documents if d["filename"] == target_name)

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [24]:
target_doc

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [25]:
doc_vec = embed.encode(target_doc["content"])
cos = float(doc_vec.dot(v))
print(f"cosine similarity = {cos:.4f}")

cosine similarity = 0.3611


Q3

In [26]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [27]:
chunks[1]

{'start': 1000,
 'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometim

In [28]:
chunk_texts = [c["content"] for c in chunks]
batch_size = 50
X = []
for i in tqdm(range(0, len(chunk_texts), batch_size)):
    X.extend(embed.encode_batch(chunk_texts[i:i + batch_size]))
X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [29]:
len(chunk_texts)

295

In [30]:
X.shape

(295, 384)

In [31]:
scores = X.dot(v)

In [32]:
scores[94]


np.float64(0.6489017718578813)

In [33]:
best = int(np.argmax(scores))
best

94

In [34]:
chunks[best]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [35]:
print(f"top chunk filename = {chunks[best]['filename']} (score={scores[best]:.4f})")

top chunk filename = 02-vector-search/lessons/07-sqlitesearch-vector.md (score=0.6489)


Q4

In [36]:
X[5].shape

(384,)

In [37]:
chunks[5]

{'start': 2000,
 'content': '\nThe safest way to store the key is in a `.env` file that never gets\ncommitted to git.\n\nCreate a `.env` file in your project folder and put your API key in\nit:\n\n```bash\nOPENAI_API_KEY=sk-YOUR_KEY_HERE\n```\n\nNow add `.env` to `.gitignore` to make sure you never accidentally\ncommit your key:\n\n```bash\n.env\n```\n\nNever commit `.env` to git. Treat the API key like a password. If it\nleaks, someone else can run up charges on your account.\n\n## Starting Jupyter\n\nStart Jupyter:\n\n```bash\nuv run jupyter notebook\n```\n\nCreate a new notebook. Throughout the course, you\'ll copy code from\nthe section notes into notebook cells.\n\nCheck that the OpenAI client works:\n\n```python\nfrom dotenv import load_dotenv\nload_dotenv()\n\nfrom openai import OpenAI\nopenai_client = OpenAI()\n```\n\nIf you see an error, make sure the key in your `.env` file is\ncorrect.\n\nFor Groq or other OpenAI-compatible providers, add the key to\n`.env`:\n\n```bash\nGROQ

In [38]:
chunks[5]["content"][:500]  # выводим первые 500 символов текста пятого чанка

'\nThe safest way to store the key is in a `.env` file that never gets\ncommitted to git.\n\nCreate a `.env` file in your project folder and put your API key in\nit:\n\n```bash\nOPENAI_API_KEY=sk-YOUR_KEY_HERE\n```\n\nNow add `.env` to `.gitignore` to make sure you never accidentally\ncommit your key:\n\n```bash\n.env\n```\n\nNever commit `.env` to git. Treat the API key like a password. If it\nleaks, someone else can run up charges on your account.\n\n## Starting Jupyter\n\nStart Jupyter:\n\n```bash\nuv run jupyter noteb'

In [39]:
vindex = VectorSearch()
vindex.fit(X, chunks)

In [40]:
v_q4 = embed.encode("What metric do we use to evaluate a search engine?")
v_q4

array([-3.45251120e-02, -4.76347907e-02, -9.52288143e-02,  2.40308090e-02,
       -1.80039094e-02,  3.22336786e-02,  4.11009181e-04,  2.86892654e-02,
        2.71596053e-02, -6.36475643e-02, -3.86707619e-02, -5.52097101e-02,
        3.84640827e-02,  3.63455416e-02, -9.32710316e-02, -5.29945155e-02,
        8.20461574e-02, -5.03414745e-03,  2.50910438e-02, -8.70656696e-02,
        7.20100754e-02,  1.29639035e-02,  1.16451658e-01, -8.20982256e-02,
       -6.52023347e-03, -4.00227584e-02, -8.36716391e-02, -1.24123525e-02,
       -1.90502106e-02, -6.24077402e-03, -3.73876830e-02,  6.12152591e-03,
        5.60926979e-02,  6.56479763e-02, -6.34850137e-02,  3.43527274e-02,
       -3.35941763e-02, -3.51476658e-02,  2.54281298e-02, -9.75776093e-03,
       -4.84886223e-02, -2.35025387e-02, -2.90664668e-02,  3.65187041e-02,
        2.23322831e-02, -3.29535923e-02, -4.82611746e-02,  8.59540277e-03,
       -3.70969933e-03,  5.49367302e-02, -9.72623872e-02, -1.93188061e-04,
       -1.25472844e-02, -

In [41]:
v_q4.shape

(384,)

In [42]:
vres = vindex.search(v_q4, num_results=5)
print(f"first result filename = {vres[0]['filename']}")

first result filename = 04-evaluation/lessons/05-search-metrics.md


In [43]:
vres

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

Q5

In [44]:
tindex = Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)


In [45]:
query_q5 = "How do I store vectors in PostgreSQL?"
v_q5 = embed.encode(query_q5)
v_q5.shape

(384,)

In [46]:
vector_results_q5 = vindex.search(v_q5, num_results=5)
vector_results_q5

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [47]:
text_results_q5 = tindex.search(query_q5, num_results=5)
text_results_q5

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

In [48]:
vector_files = {r["filename"] for r in vector_results_q5}
vector_files

{'02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md'}

In [49]:
text_files = {r["filename"] for r in text_results_q5}
text_files

{'02-vector-search/lessons/01-intro.md',
 '02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md'}

In [50]:
print(f"in vector but not text = {sorted(vector_files - text_files)}")

in vector but not text = ['02-vector-search/lessons/08-pgvector.md']


Q6

In [59]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    print(f"docs : {docs}")
    print(f"key : {key}")
    print(f"ranked : {ranked}")
    return [docs[key] for key in ranked[:num_results]]

In [52]:
query_q6 = "How do I give the model access to tools?"
v_q6 = embed.encode(query_q6)

In [54]:
vector_results = vindex.search(v_q6, num_results=5)
vector_results

[{'start': 2000,
  'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline ag

In [55]:
text_results = tindex.search(query_q6, num_results=5)
text_results

[{'start': 0,
  'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can

In [60]:
fused = rrf([vector_results, text_results])
fused

docs : {('01-agentic-rag/lessons/01-intro.md', 2000): {'start': 2000, 'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separ

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 

In [61]:
print(f"RRF first result filename = {fused[0]['filename']}")

RRF first result filename = 01-agentic-rag/lessons/13-function-calling.md
